# AutoDubber on Colab (GPU) — generation only

Runs the dubbing pipeline (transcribe → translate → TTS → build) on a free Colab **GPU**, then lets you download `output.mp4`. **No publishing here** — download the file and publish from your desktop app, so your YouTube/Bluesky/Zernio credentials never touch this shared VM.

**Before running:** set the runtime to GPU — `Runtime → Change runtime type → Hardware accelerator: GPU (T4)`.

**Important:** this notebook clones the code from GitHub, so the GPU support (the `WHISPER_DEVICE`/`DEMUCS_DEVICE` changes) and any fixes must be **pushed to `main` first**. The setup cell checks this for you.

The only secret needed here is your **Gemini API key** (translation, and vision if available).

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'No GPU — set Runtime > Change runtime type > GPU'
import torch
print('CUDA available:', torch.cuda.is_available())

## 2. Get the code + install dependencies

Idempotent: safe to re-run (uses absolute paths, won't nest clones). If your GitHub repo is **private**, replace the clone URL with a token one:
`https://<YOUR_GITHUB_TOKEN>@github.com/paramashivatma/autodubber.git`

In [ ]:
import os
# Absolute paths so re-running never nests clones (autodubber/autodubber/...).
%cd /content
if not os.path.isdir('/content/autodubber'):
    !git clone https://github.com/paramashivatma/autodubber.git
%cd /content/autodubber
!git pull -q
# python3-tk: app.py imports tkinter at module load even on the CLI path.
# ffmpeg: required by the pipeline.
!apt-get -qq install -y python3-tk ffmpeg >/dev/null
!pip install -r requirements.txt

# Verify the critical deps actually imported (pip can fail partway):
import importlib
missing = []
for m in ['edge_tts', 'faster_whisper', 'demucs', 'pydub', 'soundfile', 'yt_dlp']:
    try:
        importlib.import_module(m)
    except Exception as e:
        missing.append(m); print('MISSING:', m, '->', e)
print('\nDeps OK.' if not missing else '\n^ install the missing deps above, then re-run this cell.')

# Confirm the cloned code has GPU support (must be pushed to GitHub first):
has_gpu_code = 'WHISPER_DEVICE' in open('dubber/transcriber.py').read()
print('GPU-capable code present:', has_gpu_code,
      '' if has_gpu_code else '<-- push the device-gating changes to main, then re-run')

## 3. Configure GPU + API key

These env vars switch Whisper and Demucs onto the GPU (the code defaults to CPU when unset, so your desktop is unaffected).

In [ ]:
import os
from getpass import getpass

os.environ['WHISPER_DEVICE'] = 'cuda'
os.environ['WHISPER_COMPUTE_TYPE'] = 'float16'
os.environ['DEMUCS_DEVICE'] = 'cuda'
os.environ['DUB_VERIFICATION'] = '0'  # keep QA re-transcription off (faster)

# Only Gemini is needed in generation-only mode (translation + vision).
# Type the key and press ENTER in the box that appears below.
os.environ['GEMINI_API_KEY'] = getpass('Gemini API key: ').strip()
print('Configured. GPU env set, Gemini key stored for this session only.')

## 4. Choose the video + languages

In [ ]:
VIDEO_URL   = 'https://www.youtube.com/shorts/REPLACE_ME'  #@param {type:"string"}
SOURCE_LANG = 'English'   #@param {type:"string"}
TARGET_LANG = 'Gujarati'  #@param {type:"string"}
VOICE       = 'Gujarati - Niranjan (M)'  #@param {type:"string"}
print('Will dub:', VIDEO_URL, '->', TARGET_LANG, 'voice', VOICE)

## 5. Run the dub (generation only)

Calls the same headless CLI the desktop app ships (`dub_only=True`): generates `output.mp4`, skips captions/publishing. Watch the `[STATUS]` / `[PROGRESS]` lines.

In [ ]:
!python app.py "{VIDEO_URL}" output.mp4 --source-lang "{SOURCE_LANG}" --target-lang "{TARGET_LANG}" --voice "{VOICE}"

## 6. Preview + download `output.mp4`

In [ ]:
import os
assert os.path.exists('output.mp4'), 'output.mp4 not found — check the run log in cell 5 for errors.'
print('Size: %.1f MB' % (os.path.getsize('output.mp4')/1e6))

from IPython.display import HTML
from base64 import b64encode
mp4 = open('output.mp4','rb').read()
HTML('<video width=360 controls><source src="data:video/mp4;base64,%s" type="video/mp4"></video>' % b64encode(mp4).decode())

In [ ]:
from google.colab import files
files.download('output.mp4')